In [1]:
import archs4py as a4
import h5py

In [2]:
file = "/scratch/by2372/pipeline/ARCHS4_Macrophages/human_tpm_v2.latest.h5"
a4.ls(file)

data                      
│ expression            float32 | (271742, 992546)
meta                      
│ info                      
│   author                str    | ()
│   contact               str    | ()
│   creation-date         str    | ()
│   laboratory            str    | ()
│   version               str    | ()
│ samples                   
│   alignedreads          float64 | (992546,)
│   channel_count         str    | (992546,)
│   characteristics_ch1   str    | (992546,)
│   contact_address       str    | (992546,)
│   contact_city          str    | (992546,)
│   contact_country       str    | (992546,)
│   contact_institute     str    | (992546,)
│   contact_name          str    | (992546,)
│   contact_zip           str    | (992546,)
│   data_processing       str    | (992546,)
│   extract_protocol_ch1  str    | (992546,)
│   geo_accession         str    | (992546,)
│   instrument_model      str    | (992546,)
│   last_update_date      str    | (992546,)
│   library_sele

In [3]:
library_strategy  = a4.meta.field(file, "library_strategy")
lib_strat = list(set(library_strategy ))
lib_strat

['RNA-Seq']

In [4]:
library_source  = a4.meta.field(file, "library_source")
lib_source = list(set(library_source))
lib_source

['genomic',
 'transcriptomic',
 'genomic single cell',
 'other',
 'transcriptomic single cell']

In [5]:
library_selection  = a4.meta.field(file, "library_selection")
lib_selection = list(set(library_selection))
lib_selection

['size fractionation', 'RANDOM', 'CAGE', 'cDNA', 'RACE', 'other']

In [4]:
status  = a4.meta.field(file, "status")
status_list = list(set(status))
#status_list
# gives data of when the sample was public

In [3]:
organism_ch1 = a4.meta.field(file,"organism_ch1")
organisms = list(set(organism_ch1))
organisms

['Homo sapiens']

In [12]:
characteristics_ch1 = a4.meta.field(file,"characteristics_ch1")
characteristics = list(set(characteristics_ch1))
characteristics_cell_type = [
    a for a in characteristics_ch1 if "cell type" in a.lower()
]
# Comprehensive information about cell-line, tissue type, disease, etc

In [8]:
len(characteristics_ch1)

992546

In [11]:
len(characteristics_cell_type)

468229

In [15]:
molecule_ch1 = a4.meta.field(file,"molecule_ch1")
molecule = list(set(molecule_ch1))
molecule

['total RNA',
 'cytoplasmic RNA',
 'genomic DNA',
 'protein',
 'polyA RNA',
 'other',
 'nuclear RNA']

In [16]:
relation = a4.meta.field(file,"relation")
relations = list(set(relation))
#relations
# Gives SRA number

In [17]:
sample = a4.meta.field(file,"sample")
samples = list(set(sample))
#samples
# gives GSM number

In [18]:
title = a4.meta.field(file,"title")
titles = list(set(title))
#titles
# name of the specific submission?

In [19]:
t_types = a4.meta.field(file,"type")
typess = list(set(t_types))
typess

['SRA']

In [9]:
singlecellprobability = a4.meta.field(file,"singlecellprobability")
singleprob = list(set(singlecellprobability))
singleprob[0]

np.float64(0.0)

# Filtering strategy: 
## 1. Use singlecellprobability to filter out likely single-cell samples.
### If the prob is <0.5 then it is not single cell and if it is >= 0.5 then it is single cell
## 2. filter for "transcriptomic" in "library_source"
## 3. filter for "total RNA", "cytoplasmic RNA", "polyA RNA" ??? 
## 4. Use Source names to filter for macrophage types of samples 
### Osteoclasts, microglia, alveolar macrophage, kupffer cell, histiocyte, langerhans cells (dermal), monocytes, adipose (tissue) macrophages, hofbauer cells (placenta), Red pulp macrophages (spleen)
### Alveoloar, microliga, monocytes, macrophages
### Use list of words to exclude already constructed from previous filtering of ARCHS4
## 5. Use source names for disease states, cell lines, etc 
## 6. Filter for transcription factor genes

# 

## Filtering strategy 1 & 2 & 3: Filtering out single cell samples and filtering for transcriptomic samples only. Only include total RNA, cytoplasmic RNA, and polyA RNA. Have the number of samples included after each filtering show in out file. 

In [ ]:
import h5py
import sys

file = "/scratch/by2372/pipeline/ARCHS4_Macrophages/human_tpm_v2.latest.h5"

with h5py.File(file, "r") as fin, h5py.File('human_tpm_v2.2_NOSC.h5', 'w') as fout:

    # copy over the transcript IDs and gene symbols to the new h5 file
    fin.copy("meta/transcripts/ensembl_id", fout, name="meta/transcripts/ensembl_id")
    fin.copy("meta/transcripts/symbol", fout, name="meta/transcripts/symbol")

    # define a variable containing the expression data
    expr_data = fin['data/expression']

    # import source names
    source_names = fin['meta/samples/source_name_ch1'][:]

    # import library source labels
    library_source = fin['meta/samples/library_source'][:]

    # import the molecule name
    molecule = fin['meta/samples/molecule_ch1'][:]

    # import the sample characteristics
    characteristics = fin['meta/samples/characteristics_ch1'][:]

    # import the single cell probability values
    SC_prob = fin['meta/samples/singlecellprobability'][:]

    print("Finding matched indices for transcriptomic samples and non-single cell samples")
    sys.stdout.flush()

    # Define the needed metrics to find samples of interest for new H5 file
    n_genes, n_samples = expr_data.shape
    chunk_size = 2000


    # Create an empty dataset for the filtered expression in fout
    new_expr_data = fout.create_dataset(
        "data/expression",
        (n_genes, 0),                    # start with 0 samples
        maxshape=(n_genes, None),        # allow dynamic extension
        dtype=expr_data.dtype,
        chunks=True
    )

    # Create an empty dataset for the source_name in fout
    new_source = fout.create_dataset(
        "meta/samples/source_name_ch1",
        (0,),                            # start empty
        maxshape=(None,),                # allow appending
        dtype=source_names.dtype,
        chunks=True
    )

    # Create an empty dataset for characteristics_ch1 in fout
    new_char = fout.create_dataset(
        "meta/samples/characteristics_ch1",
        (0,),                            # start empty
        maxshape=(None,),                # allow appending
        dtype=characteristics.dtype,
        chunks=True
    )

    sample = 0
    for start_sample in range(0, n_samples, chunk_size):
        end_sample = min(start_sample+chunk_size, n_samples)
        chunk_lib = library_source[start_sample:end_sample]
        chunk_molecule = molecule[start_sample:end_sample]
        chunk_SC_prob = SC_prob[start_sample:end_sample]

        for i in range(end_sample - start_sample):
            lib = chunk_lib[i].lower()
            mole = chunk_molecule[i].lower()
            if (chunk_SC_prob[i] < 0.5):
                if (b"transcriptomic" in lib) and (b"single" not in lib):
                    if (b'total rna' in mole) or (b'cytoplasmic rna' in mole) or (b'polya rna' in mole) or (b'nuclear rna' in mole):
                        idx = start_sample+i
                        matched_source = source_names[idx]
                        new_source.resize((sample + 1,))
                        new_source[sample] = matched_source

                        matched_char = characteristics[idx]
                        new_char.resize((sample + 1,))
                        new_char[sample] = matched_char

                        new_expr_data.resize((n_genes, sample + 1))
                        for start_gene in range(0, n_genes, chunk_size):
                            end_gene = min(start_gene + chunk_size, n_genes)
                            chunk_expr = expr_data[start_gene:end_gene, idx]
                            new_expr_data[start_gene:end_gene,sample] = chunk_expr
        
                        sample += 1
                        print("Sample " + str(sample) + " is done.")
                        sys.stdout.flush()           
                
    print("Matched indices for transcriptomic samples have been found")
    sys.stdout.flush()

In [2]:
no_SC_file = "/scratch/by2372/pipeline/ARCHS4_Macrophages/human_tpm_v2.2_NOSC.h5"
#redownload the most recent transcript file
a4.ls(no_SC_file)

data                      
│ expression            float32 | (271742, 661840)
meta                      
│ samples                   
│   characteristics_ch1   str    | (661840,)
│   geo_accession         str    | (661840,)
│   source_name_ch1       str    | (661840,)
│ transcripts               
│   ensembl_id            str    | (271742,)
│   symbol                str    | (271742,)


In [5]:
source_name_ch1 = a4.meta.field(no_SC_file, "source_name_ch1")
# Make unique and convert each to uppercase
source_name = list(set(name.upper() for name in source_name_ch1))
# Write the list to the file
with open("all_unique_names.txt", 'w') as file:
    for item in source_name:
        file.write("%s\n" % item)

In [3]:
geo_accession = a4.meta.field(no_SC_file, "geo_accession")
# Make unique and convert each to uppercase
geo_num = list(set(name.upper() for name in geo_accession))
# Write the list to the file
with open("all_unique_geo.txt", 'w') as file:
    for item in geo_num:
        file.write("%s\n" % item)